# 28 · Long-Context vs RAG

**Idea:** modern models have huge context windows, tempting you to skip retrieval and just "stuff" the whole corpus into the prompt. This notebook compares the two head-to-head. The trade-off is **cost** and **focus**: stuffing pays for every token on every call and dilutes attention (the *lost-in-the-middle* effect), while RAG sends only what matters but can miss a passage the retriever ranked low.

> Both approaches run on Ollama. One cell optionally prices the two on OpenAI — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

**Long-context:** `stuff_answer` dumps the entire corpus into the prompt. Note the prompt-token count — you pay this on *every* question.

In [ ]:
from rag_lab import load_markdown, stuff_answer
from study_buddy import CostTracker

chunks = load_markdown()
question = 'What is reranking and when does it run?'

t_stuff = CostTracker()
stuffed = stuff_answer(question, chunks, provider='ollama', tracker=t_stuff)
print(f'[{stuffed.approach}] prompt tokens = {stuffed.prompt_tokens}')
print(stuffed.answer[:200], '...')

**RAG:** retrieve the top-k first, then answer from just those chunks. Compare the prompt size — far fewer tokens for a comparable answer.

In [ ]:
from rag_lab import NumpyStore, rag_answer
from study_buddy import count_tokens

store = NumpyStore(provider='ollama'); store.add(chunks)
t_rag = CostTracker()
res = rag_answer(question, store, provider='ollama', k=4, tracker=t_rag)
rag_tokens = count_tokens(res.context)

print(f'[rag] prompt tokens = {rag_tokens}')
print(res.answer[:200], '...')
print(f'\nstuffing used {stuffed.prompt_tokens / max(rag_tokens,1):.1f}x more context tokens than RAG')

That token gap *is* the cost gap — and it recurs on every single request. Optionally price both approaches on OpenAI to see it in USD. Skips if no key.

In [ ]:
try:
    cs = CostTracker(); cr = CostTracker()
    stuff_answer(question, chunks, provider='openai', model='gpt-4o-mini', tracker=cs)
    rag_answer(question, store, provider='openai', model='gpt-4o-mini', k=4, tracker=cr)
    print('stuffing cost:', cs.summary())
    print('rag      cost:', cr.summary())
except Exception as e:
    print('openai skipped:', type(e).__name__, e)

**Takeaway:** long-context is *simpler* (no retriever) but pays per token per call and can lose facts buried mid-prompt; RAG adds retrieval complexity but keeps prompts small, cheap, and focused. Use long-context for small/one-off corpora; use RAG when the corpus is large or queried often. They also combine — retrieve a *lot*, then let a long window sort it out.

Next up → `29_multimodal_rag.ipynb`, extending retrieval beyond text.